In [9]:
from dotenv import load_dotenv
load_dotenv()


True

In [10]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain.messages import HumanMessage

@tool
def calaculate_square(x : float) -> float :
    """Returns the square of the number """

    return x**2 ; 

@tool 
def calculate_square_root(x : float) -> float :
    """Returns square root of the number"""

    return x**0.5

model = init_chat_model(model="gemini-3.5-flash" , model_provider="google-genai")

subagent_1 = create_agent(model , tools=[calaculate_square])
subagent_2 = create_agent(model , tools=[calculate_square_root])


## Calling sub-agents 

In [11]:
@tool 
def call_sub_agent_1(x : float) -> float : 
    """Call subagent 1 in order to calculate the square  of a number"""
    response = subagent_1.invoke({"messages": [HumanMessage(content=f"Calculate the square  of {x}")]})
    return response["messages"][-1].content 

def call_sub_agent_2(x : float) -> float  : 
    """Call subagent 2 in order to calculate the square root of a number"""
    response = subagent_1.invoke({"messages": [HumanMessage(content=f"Calculate the square root of {x}")]})
    return response["messages"][-1].content


main_agent = create_agent(model , tools=[call_sub_agent_1 , call_sub_agent_2 ] , system_prompt="You are a helpful assistant who can call subagents to calculate the square root or square of a number.")

In [12]:
from pprint import pprint


question = "What is the square root of 456?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})

pprint(response)

{'messages': [HumanMessage(content='What is the square root of 456?', additional_kwargs={}, response_metadata={}, id='2454a71a-6194-4e82-af93-83c27ac75152'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'call_sub_agent_2', 'arguments': '{"x": 456}'}, '__gemini_function_call_thought_signatures__': {'rw1d3994': 'Eo0DCooDARFNMg9uVhQCoDY86RULNJdybBUCLtcFSGIAcm8yRmKKSVhbKtmtfznMjdd7n2nYkGCYECVzUFNinqU9evyKTlJcpAzXZhrZ5yYZgBQ6H2DOMiBA307b5lw+bRq8MmMhjPxaBT0lt4MhxMVcAc8e10XLUZlLdyqZnsgv6G62yVld8MwPoWdZeyJxt8ebRl8oCEe8TMe/EgUagoSZQo5zmQl5qkrhq4gEelWXkPgcPfdFcVI4WQnoLmrGpXvIAvtA+OwgNLNfJGCgqqdx1WFBgF8VvJCGxDvYVZxvm4Ymps0h/NvC+O2mV0C/83pSkNPZwrl1mTc6tzL/IBR2DHOyWxFOsou2AkI8dWKb3E06S7bts1qrcA+2wbTfl+zeL62tCounkbGW0cZ78m2nEzD6sHjX1uXPYuZkj1ofQed+SQP0MGSJacWRmf0hq3GqETNV0yw+DzfvM+crQYi5xLtQpDLU276pLkivgrBQimyQ+ePQgyEvWygR5ClsFvRUYIsnSrY03Gqk/G7pkw=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash', 'safety_ratings': [], 'model_prov